In [21]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import coalesce, col, to_date

In [22]:
spark = SparkSession.builder \
    .appName("EnrichUserProfiles") \
    .master("local[*]") \
    .getOrCreate()

In [23]:
df_customers = spark.read.parquet("silver/customers")
df_profiles = spark.read.parquet("silver/user_profiles")

In [24]:
df_enriched = df_customers.alias("c") \
    .join(
        df_profiles.alias("p"),
        on="email",
        how="left"
    ) \
    .select(
        col("c.client_id"),
        coalesce(col("c.first_name"), col("p.first_name")).alias("first_name"),
        coalesce(col("c.last_name"), col("p.last_name")).alias("last_name"),
        coalesce(col("c.state"), col("p.state")).alias("state"),
        col("c.email"),
        col("c.registration_date"),
        col("p.birth_date"),
        col("p.phone_number")
    )

In [25]:
df_enriched.show(10, truncate=False)
df_enriched.printSchema()

+---------+----------+----------+-----+----------------------------+-----------------+----------+---------------------+
|client_id|first_name|last_name |state|email                       |registration_date|birth_date|phone_number         |
+---------+----------+----------+-----+----------------------------+-----------------+----------+---------------------+
|26       |Sarah     |Villanueva|Idaho|sarah_villanueva@example.com|2022-08-03       |2000-11-28|551-590-0422x94568   |
|27       |Kimberly  |Myers     |Utah |kimberly_myers@example.com  |2022-08-01       |1969-08-31|(417)304-2814        |
|28       |Desiree   |Cain      |Maine|desiree_cain@example.com    |2022-08-05       |1986-11-21|(546)118-7755x1717   |
|31       |Whitney   |Stark     |Idaho|whitney_stark@example.com   |2022-08-05       |2004-05-20|(349)263-5110x873    |
|34       |Faith     |Cabrera   |Maine|faith_cabrera@example.com   |2022-08-04       |1995-04-11|+1-577-389-3055x50824|
|44       |Matthew   |Bell      |Texas|m

In [26]:
os.makedirs("gold", exist_ok=True)
df_enriched.write.mode("overwrite").parquet("gold/user_profiles_enriched")